In [2]:

from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    r"C:\Users\crime\OneDrive\Desktop\RAG-COURSE\Data",
    glob="**/*.md",
    exclude=[
        "README.md",
        "chatbot_instructions.md",
        "sources_and_verification.md"
    ],
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")


Loaded 7 documents


C:\Users\crime\AppData\Local\Temp\ipykernel_23448\2081545035.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [3]:
documents

[Document(metadata={'source': 'C:\\Users\\crime\\OneDrive\\Desktop\\RAG-COURSE\\Data\\certifications.md'}, page_content='# Certifications and Professional Learning\n\n## Certifications visible on the public LinkedIn profile\n\n- **Advanced React.js** — Meta; issued December 2023. Certificate ID shown publicly: GTA4C7TABYRS.\n- **Mastering Laravel Framework and PHP** — Board Infinity; issued December 2023. Certificate ID shown publicly: 7KRGUE6V8D2R.\n- **DevOps Certification** — Techedge Group; issued November 2023. Certificate ID shown publicly: 655807.\n- **Scrum Certificate** — CertiProf International. Certificate ID shown publicly: 91509495; issue date not visible in the indexed profile.\n\n## Microsoft Power BI learning milestone\n\nA public LinkedIn post describes completing the sixth course in the Microsoft Power BI Data Analyst Professional Certificate sequence: **Creative Designing in Power BI**, with a stated duration of 20 hours. The post says this was progress toward the PL

In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers = [("#", "h1"), ("##", "h2"), ("###", "h3")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)

chunks = []
for doc in documents:
    for piece in md_splitter.split_text(doc.page_content):
        piece.metadata["source"] = doc.metadata["source"]  
        chunks.append(piece)

In [6]:
len(chunks)

47

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\crime\OneDrive\Desktop\RAG-COURSE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class EmbeddingModel:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()  # fixed: now matches the method name below

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name} ...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: "
                  f"{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,  
        )
        print(f"Done. Embeddings shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimensions(self) -> int:  # fixed typo in the name
        if self.model is None:
            raise ValueError("Model is not loaded.")
        return self.model.get_sentence_embedding_dimension()

In [9]:
embedder = EmbeddingModel()
texts = ["Customers can request a refund within 30 days.", "Gift cards are not refundable."]
vectors = embedder.generate_embeddings(texts)

Loading embedding model: all-MiniLM-L6-v2 ...


c:\Users\crime\OneDrive\Desktop\RAG-COURSE\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\crime\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\crime\OneDrive\Desktop\RAG-COURSE\.venv\Lib\site-packages\huggingface_hu

Model loaded successfully. Embedding dimension: 384
Generating embeddings for 2 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.27it/s]

Done. Embeddings shape: (2, 384)


array([[-6.96437955e-02,  9.73175745e-03,  8.85130167e-02,
        -9.70305817e-04, -1.19726006e-02, -1.48484912e-02,
        -3.67641486e-02, -1.84410494e-02, -1.18076568e-02,
         3.01270504e-02,  7.25398809e-02,  2.44177617e-02,
        -5.90984989e-03, -8.47906545e-02, -3.82586382e-02,
        -5.71621768e-02,  4.95865336e-03,  1.48093579e-02,
        -2.41800584e-03,  6.46753097e-03,  1.35596450e-02,
        -4.82225083e-02, -4.14539278e-02, -7.50680338e-04,
         4.73724119e-02,  2.77701467e-02, -1.78584810e-02,
         5.26970103e-02, -4.93805408e-02, -1.61442570e-02,
         3.21465582e-02,  2.51829326e-02, -3.84238847e-02,
        -3.74945439e-02, -5.19576250e-03, -9.74225998e-03,
        -8.29887763e-02, -3.17200385e-02,  4.53244755e-03,
        -7.68247480e-03, -9.24210176e-02,  2.39683911e-02,
        -5.20851910e-02,  3.13964188e-02,  6.26948476e-02,
        -4.08668220e-02,  6.24307198e-03,  1.52963996e-02,
         6.61552176e-02,  9.41314846e-02, -2.50893496e-0